In [ ]:
import pandas as pd
import numpy as np
import torch
import esm
import os
# from esm2_encoder import encode_AA_sequences

script_dir = os.path.dirname(os.path.abspath("3a-esm2_encoder.ipynb"))

In [ ]:
data_path = os.path.join(script_dir, "../data", "unique_epitopes.tsv")
unique_epitopes = pd.read_csv(data_path, sep='\t')

seqs = unique_epitopes['Epitope'].astype(str).tolist()

In [ ]:
def encode_AA_sequences(sequences, model_name="esm2_t33_650M_UR50D", device='cuda'):
    # Encode peptide sequences with ESM-2 and return mean embeddings (shape: n_seqs x embed_dim)
    # Load model directly from the installed package
    print(f"Loading {model_name}...")
    model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
    
    batch_converter = alphabet.get_batch_converter()
    model.eval()
    
    # Move to device
    if device == 'cuda' and torch.cuda.is_available():
        model = model.cuda()
        print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    else:
        device = 'cpu'
        print("Using CPU")
    
    # Prepare data - ESM expects list of tuples (label, sequence)
    data = [(f"seq_{i}", seq) for i, seq in enumerate(sequences)]
    
    # Convert sequences to tokens
    batch_labels, batch_strs, batch_tokens = batch_converter(data)
    batch_lens = (batch_tokens != alphabet.padding_idx).sum(1)
    
    # Move to device
    if device == 'cuda':
        batch_tokens = batch_tokens.cuda()
    
    # Extract embeddings
    print(f"Encoding {len(sequences)} sequences...")
    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[model.num_layers], return_contacts=False)
    
    # Get representations from last layer
    token_representations = results["representations"][model.num_layers]
    
    # Calculate mean embeddings (exclude BOS and EOS tokens)
    embeddings = []
    for i, tokens_len in enumerate(batch_lens):
        # Take mean over sequence length (skip BOS at position 0 and EOS at the end)
        seq_embedding = token_representations[i, 1:tokens_len-1].mean(0)
        embeddings.append(seq_embedding.cpu().numpy())
    
    embeddings = np.array(embeddings)
    print(f"Done! Embedding shape: {embeddings.shape}")
    
    return embeddings

In [ ]:
embeddings_epitope = encode_AA_sequences(seqs, device='cpu')

from pathlib import Path

out_dir = Path("../data")
out_dir.mkdir(exist_ok=True)
np.save(out_dir / "epitope_esm2_embeddings.npy", embeddings_epitope)